In [1]:
# ============================================================
# EXPERIMENT 5
# EcoBotX-Light + P2
# NO CBAM
# NO KNOWLEDGE DISTILLATION
# ============================================================

from pathlib import Path
import os
import yaml
import torch

from ultralytics import YOLO

print("=" * 70)
print("EXPERIMENT 5")
print("EcoBotX-Light + P2 Small-Object Detection Head")
print("CBAM: OFF")
print("Knowledge Distillation: OFF")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

BASE_DIR = Path(r"G:\EcoBotX_YOLO_training")

EXP_DIR = BASE_DIR / "experiment5_ecobotx_p2"

EXP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_YAML = EXP_DIR / "experiment5_ecobotx_p2.yaml"

DATASET_YAML = Path(r"G:\EcoBotX_YOLO\dataset.yaml")

print(f"Experiment directory : {EXP_DIR}")
print(f"Model YAML           : {MODEL_YAML}")
print(f"Dataset YAML         : {DATASET_YAML}")

# ------------------------------------------------------------
# Hardware
# ------------------------------------------------------------

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print(f"Device               : {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")

EXPERIMENT 5
EcoBotX-Light + P2 Small-Object Detection Head
CBAM: OFF
Knowledge Distillation: OFF
Experiment directory : G:\EcoBotX_YOLO_training\experiment5_ecobotx_p2
Model YAML           : G:\EcoBotX_YOLO_training\experiment5_ecobotx_p2\experiment5_ecobotx_p2.yaml
Dataset YAML         : G:\EcoBotX_YOLO\dataset.yaml
Device               : 0
GPU                  : NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
# ============================================================
# SECTION 2 — DATASET CHECK
# ============================================================

if not DATASET_YAML.exists():
    raise FileNotFoundError(
        f"Dataset YAML not found:\n{DATASET_YAML}"
    )

with open(DATASET_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

# ------------------------------------------------------------
# Get class names
# ------------------------------------------------------------

if "names" not in data_cfg:
    raise KeyError(
        "The dataset.yaml does not contain a 'names' field."
    )

class_names = data_cfg["names"]

# Handle both:
# names:
#   - bottle
#   - can
#
# and:
# names:
#   0: bottle
#   1: can

if isinstance(class_names, dict):
    class_names = list(class_names.values())

elif isinstance(class_names, list):
    class_names = class_names

else:
    raise TypeError(
        "Unsupported format for 'names' in dataset.yaml."
    )

# ------------------------------------------------------------
# Calculate number of classes automatically
# ------------------------------------------------------------

NUM_CLASSES = len(class_names)

print("\nClasses:")

for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

print(f"\nNumber of classes: {NUM_CLASSES}")

# ------------------------------------------------------------
# Verify EcoBotX has four classes
# ------------------------------------------------------------

if NUM_CLASSES != 4:
    raise ValueError(
        f"Expected 4 classes, but found {NUM_CLASSES}."
    )

print("\n[OK] Dataset configuration verified.")

DATASET INFORMATION

Classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

Number of classes: 4

[OK] Dataset configuration verified.


In [5]:
# ============================================================
# SECTION 3 — EXPERIMENT 5 YAML
# EcoBotX-Light + P2
# NO CBAM
# NO KNOWLEDGE DISTILLATION
# ============================================================

yaml_content = """
nc: 4

depth_multiple: 0.33
width_multiple: 0.25

backbone:

  # ----------------------------------------------------------
  # P1
  # ----------------------------------------------------------

  - [-1, 1, Conv, [64, 3, 2]]          # 0

  # ----------------------------------------------------------
  # P2 / 4
  # ----------------------------------------------------------

  - [-1, 1, Conv, [128, 3, 2]]         # 1
  - [-1, 3, C2f, [128, True]]          # 2

  # ----------------------------------------------------------
  # P3 / 8
  # ----------------------------------------------------------

  - [-1, 1, Conv, [256, 3, 2]]         # 3
  - [-1, 6, C2f, [256, True]]          # 4

  # ----------------------------------------------------------
  # P4 / 16
  # ----------------------------------------------------------

  - [-1, 1, Conv, [512, 3, 2]]         # 5
  - [-1, 6, C2f, [512, True]]          # 6

  # ----------------------------------------------------------
  # P5 / 32
  # ----------------------------------------------------------

  - [-1, 1, Conv, [1024, 3, 2]]        # 7
  - [-1, 3, C2f, [1024, True]]         # 8

  # ----------------------------------------------------------
  # SPPF
  # ----------------------------------------------------------

  - [-1, 1, SPPF, [1024, 5]]            # 9


head:

  # ==========================================================
  # P5 -> P4
  # ==========================================================

  - [-1, 1, nn.Upsample, [None, 2, nearest]]   # 10
  - [[-1, 6], 1, Concat, [1]]                  # 11
  - [-1, 3, C2f, [512]]                        # 12

  # ==========================================================
  # P4 -> P3
  # ==========================================================

  - [-1, 1, nn.Upsample, [None, 2, nearest]]   # 13
  - [[-1, 4], 1, Concat, [1]]                  # 14
  - [-1, 3, C2f, [256]]                        # 15

  # ==========================================================
  # P3 -> P2
  # ==========================================================

  - [-1, 1, nn.Upsample, [None, 2, nearest]]   # 16
  - [[-1, 2], 1, Concat, [1]]                  # 17
  - [-1, 3, C2f, [128]]                        # 18

  # ==========================================================
  # P2 -> P3
  # ==========================================================

  - [-1, 1, Conv, [128, 3, 2]]                 # 19
  - [[-1, 15], 1, Concat, [1]]                 # 20
  - [-1, 3, C2f, [256]]                        # 21

  # ==========================================================
  # P3 -> P4
  # ==========================================================

  - [-1, 1, Conv, [256, 3, 2]]                 # 22
  - [[-1, 12], 1, Concat, [1]]                 # 23
  - [-1, 3, C2f, [512]]                        # 24

  # ==========================================================
  # P4 -> P5
  # ==========================================================

  - [-1, 1, Conv, [512, 3, 2]]                 # 25
  - [[-1, 9], 1, Concat, [1]]                  # 26
  - [-1, 3, C2f, [1024]]                       # 27

  # ==========================================================
  # FOUR-SCALE DETECTION
  # P2, P3, P4, P5
  # ==========================================================

  - [[18, 21, 24, 27], 1, Detect, [nc]]       # 28
"""

with open(MODEL_YAML, "w") as f:
    f.write(yaml_content)

print("=" * 70)
print("EXPERIMENT 5 YAML CREATED")
print("=" * 70)

print(f"\nSaved to:")
print(MODEL_YAML)

EXPERIMENT 5 YAML CREATED

Saved to:
G:\EcoBotX_YOLO_training\experiment5_ecobotx_p2\experiment5_ecobotx_p2.yaml


In [6]:
# ============================================================
# SECTION 4 — VERIFY YAML
# ============================================================

print("=" * 70)
print("EXPERIMENT 5 YAML CONTENT")
print("=" * 70)

print(MODEL_YAML.read_text())

EXPERIMENT 5 YAML CONTENT

nc: 4

depth_multiple: 0.33
width_multiple: 0.25

backbone:

  # ----------------------------------------------------------
  # P1
  # ----------------------------------------------------------

  - [-1, 1, Conv, [64, 3, 2]]          # 0

  # ----------------------------------------------------------
  # P2 / 4
  # ----------------------------------------------------------

  - [-1, 1, Conv, [128, 3, 2]]         # 1
  - [-1, 3, C2f, [128, True]]          # 2

  # ----------------------------------------------------------
  # P3 / 8
  # ----------------------------------------------------------

  - [-1, 1, Conv, [256, 3, 2]]         # 3
  - [-1, 6, C2f, [256, True]]          # 4

  # ----------------------------------------------------------
  # P4 / 16
  # ----------------------------------------------------------

  - [-1, 1, Conv, [512, 3, 2]]         # 5
  - [-1, 6, C2f, [512, True]]          # 6

  # ------------------------------------------------------

In [7]:
# ============================================================
# SECTION 5 — BUILD EXPERIMENT 5 MODEL
# ============================================================

print("=" * 70)
print("BUILDING EXPERIMENT 5")
print("EcoBotX-Light + P2")
print("=" * 70)

model = YOLO(str(MODEL_YAML))

print()
print("[OK] MODEL OBJECT CREATED")

BUILDING EXPERIMENT 5
EcoBotX-Light + P2

[OK] MODEL OBJECT CREATED


In [8]:
# ============================================================
# SECTION 6 — MODEL INFORMATION
# ============================================================

print("=" * 70)
print("EXPERIMENT 5 MODEL INFORMATION")
print("=" * 70)

model.info(verbose=True)

EXPERIMENT 5 MODEL INFORMATION
experiment5_ecobotx_p2 summary: 161 layers, 2,927,088 parameters, 2,927,072 gradients, 12.4 GFLOPs


(161, 2927088, 2927072, 12.3590144)

In [9]:
# ============================================================
# SECTION 7 — PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in model.model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.model.parameters()
    if p.requires_grad
)

print("=" * 70)
print("EXPERIMENT 5 PARAMETER INFORMATION")
print("=" * 70)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

EXPERIMENT 5 PARAMETER INFORMATION
Total parameters     : 2,927,088
Trainable parameters : 2,927,072


In [10]:
# ============================================================
# SECTION 8 — MODEL SANITY CHECK
# ============================================================

print("=" * 70)
print("EXPERIMENT 5 — MODEL SANITY CHECK")
print("=" * 70)

print("\nRunning dummy forward pass...")

# Move model to selected device
model.model.to(DEVICE)

# Create dummy 640x640 RGB image
dummy = torch.zeros(
    1,
    3,
    640,
    640,
    device=DEVICE
)

# Forward pass
with torch.no_grad():
    output = model.model(dummy)

print("[OK] Forward pass successful.")

print(f"\nDevice: {DEVICE}")

print("=" * 70)
print("SANITY CHECK PASSED")
print("=" * 70)

EXPERIMENT 5 — MODEL SANITY CHECK

Running dummy forward pass...
[OK] Forward pass successful.

Device: 0
SANITY CHECK PASSED


In [11]:
# ============================================================
# SECTION 9 — VERIFY DETECTION HEAD
# ============================================================

print("=" * 70)
print("VERIFYING DETECTION HEAD")
print("=" * 70)

detect_layer = model.model.model[-1]

print(f"Detection layer : {type(detect_layer).__name__}")

if hasattr(detect_layer, "nl"):
    print(f"Detection scales: {detect_layer.nl}")

if hasattr(detect_layer, "stride"):
    print(f"Strides         : {detect_layer.stride}")

print("=" * 70)

VERIFYING DETECTION HEAD
Detection layer : Detect
Detection scales: 4
Strides         : tensor([ 4.,  8., 16., 32.], device='cuda:0')


In [12]:
# ============================================================
# SECTION 10 — GPU MEMORY CHECK
# ============================================================

print("=" * 70)
print("GPU STATUS")
print("=" * 70)

if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved = torch.cuda.memory_reserved() / (1024 ** 3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {total:.2f} GB")
    print(f"Allocated VRAM  : {allocated:.2f} GB")
    print(f"Reserved VRAM   : {reserved:.2f} GB")

else:
    print("CUDA is not available.")

print("=" * 70)

GPU STATUS
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU
Total VRAM      : 4.00 GB
Allocated VRAM  : 0.03 GB
Reserved VRAM   : 0.08 GB


In [13]:
# ============================================================
# SECTION 11 — TRAINING CONFIGURATION
# ============================================================

TRAIN_EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 8

PROJECT_DIR = BASE_DIR / "experiment5_training"

RUN_NAME = "experiment5_ecobotx_p2"

print("=" * 70)
print("EXPERIMENT 5 TRAINING CONFIGURATION")
print("=" * 70)

print(f"Epochs       : {TRAIN_EPOCHS}")
print(f"Image size   : {IMAGE_SIZE}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Device       : {DEVICE}")
print(f"Project      : {PROJECT_DIR}")
print(f"Run name     : {RUN_NAME}")

EXPERIMENT 5 TRAINING CONFIGURATION
Epochs       : 100
Image size   : 640
Batch size   : 8
Device       : 0
Project      : G:\EcoBotX_YOLO_training\experiment5_training
Run name     : experiment5_ecobotx_p2


In [14]:
# ============================================================
# SECTION 12 — TRAIN EXPERIMENT 5
# ============================================================

print("=" * 70)
print("STARTING EXPERIMENT 5 TRAINING")
print("=" * 70)

results = model.train(
    data=str(DATASET_YAML),

    epochs=TRAIN_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,

    device=DEVICE,

    workers=4,

    pretrained=False,

    optimizer="auto",

    patience=20,

    save=True,
    save_period=10,

    plots=True,

    project=str(PROJECT_DIR),
    name=RUN_NAME,

    exist_ok=True,

    verbose=True
)

print("=" * 70)
print("EXPERIMENT 5 TRAINING FINISHED")
print("=" * 70)

STARTING EXPERIMENT 5 TRAINING
New https://pypi.org/project/ultralytics/8.4.129 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=G:\EcoBotX_YOLO\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.

In [15]:
# ============================================================
# SECTION 13 — LOCATE BEST MODEL
# ============================================================

RUN_DIR = PROJECT_DIR / RUN_NAME

BEST_MODEL = RUN_DIR / "weights" / "best.pt"
LAST_MODEL = RUN_DIR / "weights" / "last.pt"

print("=" * 70)
print("EXPERIMENT 5 MODEL FILES")
print("=" * 70)

print(f"Run directory : {RUN_DIR}")
print(f"Best model    : {BEST_MODEL}")
print(f"Last model    : {LAST_MODEL}")

if BEST_MODEL.exists():
    print("\n[OK] best.pt found.")
else:
    print("\n[WARNING] best.pt not found yet.")

EXPERIMENT 5 MODEL FILES
Run directory : G:\EcoBotX_YOLO_training\experiment5_training\experiment5_ecobotx_p2
Best model    : G:\EcoBotX_YOLO_training\experiment5_training\experiment5_ecobotx_p2\weights\best.pt
Last model    : G:\EcoBotX_YOLO_training\experiment5_training\experiment5_ecobotx_p2\weights\last.pt

[OK] best.pt found.


In [16]:
# ============================================================
# SECTION 14 — LOAD BEST MODEL
# ============================================================

best_model = YOLO(str(BEST_MODEL))

print("=" * 70)
print("BEST MODEL LOADED")
print("=" * 70)

best_model.info(verbose=True)

BEST MODEL LOADED
experiment5_ecobotx_p2 summary: 161 layers, 2,927,088 parameters, 0 gradients, 12.4 GFLOPs


(161, 2927088, 0, 12.3590144)

In [17]:
# ============================================================
# SECTION 15 — VALIDATION
# ============================================================

print("=" * 70)
print("EXPERIMENT 5 VALIDATION")
print("=" * 70)

val_results = best_model.val(
    data=str(DATASET_YAML),
    split="val",
    imgsz=640,
    batch=8,
    device=DEVICE,
    plots=True,
    save_json=True
)

print("=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

EXPERIMENT 5 VALIDATION
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
experiment5_ecobotx_p2 summary (fused): 91 layers, 2,921,568 parameters, 0 gradients, 12.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 53.918.2 MB/s, size: 6.3 KB)
val: Scanning G:\EcoBotX_YOLO\labels\val.cache... 891 images, 179 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 891/891  0.0s
val: G:\EcoBotX_YOLO\images\val\CAN (889).jpg.6uhfctqh.ingestion-846dcbfcdb-kvjk5.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.164062]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 112/112 8.7it/s 12.9s0.1s
                   all        890        713      0.935      0.913      0.961      0.747
                BOTTLE        163        163      0.949       0.92      0.979       0.76
                   CAN        190        192        0.9      0.812      0.904      0.651

In [18]:
# ============================================================
# SECTION 16 — VALIDATION METRICS
# ============================================================

val_metrics = val_results.box

print("=" * 70)
print("EXPERIMENT 5 VALIDATION RESULTS")
print("=" * 70)

print(f"Precision     : {val_metrics.mp:.4f}")
print(f"Recall        : {val_metrics.mr:.4f}")
print(f"mAP50         : {val_metrics.map50:.4f}")
print(f"mAP50-95      : {val_metrics.map:.4f}")

EXPERIMENT 5 VALIDATION RESULTS
Precision     : 0.9347
Recall        : 0.9127
mAP50         : 0.9608
mAP50-95      : 0.7467


In [19]:
# ============================================================
# SECTION 17 — TEST SET EVALUATION
# ============================================================

print("=" * 70)
print("EXPERIMENT 5 TEST EVALUATION")
print("=" * 70)

test_results = best_model.val(
    data=str(DATASET_YAML),
    split="test",
    imgsz=640,
    batch=8,
    device=DEVICE,
    plots=True,
    save_json=True
)

test_metrics = test_results.box

print("=" * 70)
print("EXPERIMENT 5 TEST RESULTS")
print("=" * 70)

print(f"Precision     : {test_metrics.mp:.4f}")
print(f"Recall        : {test_metrics.mr:.4f}")
print(f"mAP50         : {test_metrics.map50:.4f}")
print(f"mAP50-95      : {test_metrics.map:.4f}")

EXPERIMENT 5 TEST EVALUATION
Ultralytics 8.4.126  Python-3.12.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
WARNING val: Slow image access detected (ping: 0.40.1 ms, read: 0.70.2 MB/s, size: 6.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning G:\EcoBotX_YOLO\labels\test... 1115 images, 216 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1115/1115 395.8it/s 2.8s0.0s
val: New cache created: G:\EcoBotX_YOLO\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 7.7it/s 18.3s0.1s
                   all       1115        899      0.931      0.932      0.967      0.632
                BOTTLE        217        217      0.981      0.948      0.987      0.576
                   CAN        228        228      0.861      0.842      0.916      0.604
                 PAPER        250        250